# Etherpad Pad 内容重建查看器

本工具用于：
1. 查询数据库获取所有 room-id 和版本范围
2. SSH 连接到服务器执行重建脚本
3. 获取并展示重建的内容
4. 支持编辑和对比

---

## 📋 功能清单

- ✅ 查询 MySQL 数据库获取所有 Pad 列表
- ✅ 显示每个 Pad 的版本范围
- ✅ SSH 连接到远程服务器执行重建脚本
- ✅ 查看特定版本的详细内容


## 使用说明

### 完整使用流程

1. **安装依赖** - 运行第1个单元格
2. **导入配置** - 运行第2个单元格
3. **定义函数** - 运行第3-4个单元格
4. **查询 Pad 列表** - 运行第5个单元格,查看所有可用的 Pad
5. **执行重建** - 修改第6个单元格的参数,然后运行
6. **查看结果** - 运行后续单元格查看详细内容

### 参数说明

```python
# 第6个单元格 - 执行重建
pad_id = 'room-229'    # 从查询结果中选择 Pad ID
start_rev = 0          # 起始版本号（通常从0开始）
end_rev = 10           # 结束版本号

# 第9个单元格 - 查看特定版本
view_revision = 5      # 要查看的版本号

# 第10个单元格 - 版本对比
compare_rev1 = 5       # 对比版本1
compare_rev2 = 6       # 对比版本2
```

### 示例工作流

```python
# 1. 找到你感兴趣的 Pad
# 运行第5个单元格后，从输出表格中找到 pad_id

# 2. 重建内容
pad_id = 'room-229'
start_rev = 0
end_rev = 50
# 运行第6个单元格

# 3. 查看特定版本
view_revision = 10
# 运行第9个单元格

# 4. 对比版本
compare_rev1 = 10
compare_rev2 = 20
# 运行第10个单元格

# 5. 导出结果
# 运行第11个单元格
```

## 1. 安装依赖包

首次使用需要安装必要的 Python 包


In [ ]:
# 安装必要的 Python 包
#!pip install pymysql paramiko pandas ipywidgets -q


## 2. 导入库和配置


In [1]:
import pymysql
import json
import pandas as pd
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 数据库配置
DB_CONFIG = {
    'host': '112.74.92.135',
    'user': 'root',
    'password': '1q2w3e4R',
    'database': 'alic',
    'charset': 'utf8mb4',
    'port': 3306
}

# 服务器配置（用于 HTTP 请求）
SSH_CONFIG = {
    'host': '8.138.89.124',  # 服务器地址
    'port': 22,
    'username': 'root',
    'password': 'Alic2025'
}

# HTTP API 配置（使用主服务器端口，已集成到主服务器）
#HTTP_API_CONFIG = {
#    'host': '8.138.89.124',  # 服务器地址
#    'port': 9001  # 主服务器端口（默认 9001，与 Etherpad 主服务共享端口）
#}

HTTP_API_CONFIG = {
    'host': 'localhost',  # 服务器地址
    'port': 9002  # 主服务器端口（默认 9001，与 Etherpad 主服务共享端口）
}

print("✅ 配置加载完成（HTTP Stream API 已集成到主服务器，共享端口）")


✅ 配置加载完成（HTTP Stream API 已集成到主服务器，共享端口）


## 3. 定义数据库查询函数


In [2]:
def get_all_room_ids_with_versions():
    """从 store 表查询所有 room-id 及其版本范围"""
    try:
        connection = pymysql.connect(**DB_CONFIG)
        cursor = connection.cursor(pymysql.cursors.DictCursor)
        
        query = """
        SELECT 
            SUBSTRING_INDEX(SUBSTRING_INDEX(`key`, ':', 2), ':', -1) as pad_id,
            MIN(CAST(SUBSTRING_INDEX(`key`, ':', -1) AS UNSIGNED)) as min_revision,
            MAX(CAST(SUBSTRING_INDEX(`key`, ':', -1) AS UNSIGNED)) as max_revision,
            COUNT(*) as revision_count
        FROM store
        WHERE `key` LIKE 'pad:%:revs:%'
        GROUP BY pad_id
        ORDER BY pad_id
        """
        
        cursor.execute(query)
        results = cursor.fetchall()
        
        cursor.close()
        connection.close()
        
        return results
        
    except Exception as e:
        print(f"❌ 数据库查询失败: {e}")
        return []

print("✅ 数据库查询函数定义完成")


✅ 数据库查询函数定义完成


## 4. 定义 SSH 远程执行函数


In [3]:
import base64
import urllib.request
import urllib.parse

def decode_base64_field(value):
    """解码 Base64 字段"""
    if not value:
        return ''
    try:
        return base64.b64decode(value).decode('utf-8')
    except Exception:
        return value  # 如果解码失败，返回原值

def execute_rebuild_script(pad_id, start_rev=None, end_rev=None, use_base64=False):
    """
    通过 HTTP Stream 获取重建数据（避免缓冲区限制）
    
    参数:
        pad_id: Pad ID
        start_rev: 起始版本号
        end_rev: 结束版本号
        use_base64: 是否使用 Base64 编码模式（推荐用于大数据量）
        use_base64: 是否使用 Base64 编码模式（推荐用于大数据量）
    """
    try:
        # 构建 HTTP 请求 URL
        params = {
            'padId': pad_id,
            'useBase64': 'true' if use_base64 else 'false'
        }
        
        if start_rev is not None:
            params['startRev'] = str(start_rev)
        if end_rev is not None:
            params['endRev'] = str(end_rev)
        
        url = f"http://{HTTP_API_CONFIG['host']}:{HTTP_API_CONFIG['port']}/api/rebuild?{urllib.parse.urlencode(params)}"
        
        print(f"🌐 HTTP Stream 请求: {url}")
        if use_base64:
            print("📦 使用 Base64 编码模式")
        else:
            print("📝 使用标准模式")
        
        # 发送 HTTP 请求并流式读取响应
        req = urllib.request.Request(url)
        
        with urllib.request.urlopen(req, timeout=600) as response:
            print(f"✅ HTTP 连接成功 (状态码: {response.getcode()})")
            
            # 解析 NDJSON 流式响应
            all_versions = []
            header_info = None
            statistics = None
            
            # 逐行读取 NDJSON 格式的响应
            for line_bytes in response:
                line = line_bytes.decode('utf-8').strip()
                if not line:
                    continue
                
                try:
                    data = json.loads(line)
                    
                    # 第一行是头部信息
                    if 'stream' in data and data.get('stream'):
                        header_info = data
                        print(f"📊 处理范围: {data.get('requested_range', {})}")
                        continue
                    
                    # 最后一行是统计信息
                    if '_statistics' in data:
                        statistics = data.get('_statistics', {})
                        continue
                    
                    # 版本数据
                    if 'revision' in data:
                        all_versions.append(data)
                        
                except json.JSONDecodeError as e:
                    print(f"⚠️ JSON 解析警告（跳过该行）: {e}")
                    continue
            
            # 构建结果对象（兼容旧格式）
            result = {
                'success': True,
                'pad_id': pad_id,
                'head_revision': header_info.get('head_revision') if header_info else None,
                'requested_range': header_info.get('requested_range', {}) if header_info else {},
                'statistics': statistics or {},
                'versions': all_versions
            }
            
            if header_info and header_info.get('encoding') == 'base64':
                result['encoding'] = 'base64'
                print("🔓 解码 Base64 数据...")
                for version in all_versions:
                    if 'content_base64' in version:
                        version['content'] = decode_base64_field(version['content_base64'])
                        del version['content_base64']
                    if 'changeset_base64' in version:
                        version['changeset'] = decode_base64_field(version['changeset_base64'])
                        del version['changeset_base64']
                    if 'attribs_base64' in version:
                        version['attribs'] = decode_base64_field(version['attribs_base64'])
                        del version['attribs_base64']
                print("✅ Base64 解码完成")
            
            print(f"✅ 成功获取 {len(all_versions)} 个版本数据")
            return result
            
    except urllib.error.HTTPError as e:
        error_body = e.read().decode('utf-8') if e.fp else ''
        print(f"❌ HTTP 错误: {e.code} {e.reason}")
        try:
            error_data = json.loads(error_body)
            return {'success': False, 'error': error_data.get('error', str(e))}
        except:
            return {'success': False, 'error': f'HTTP {e.code}: {e.reason}'}
            
    except urllib.error.URLError as e:
        print(f"❌ URL 错误: {e.reason}")
        return {'success': False, 'error': f'Connection error: {e.reason}'}
        
    except Exception as e:
        print(f"❌ HTTP 请求失败: {e}")
        import traceback
        traceback.print_exc()
        return {'success': False, 'error': str(e)}

print("✅ HTTP Stream 执行函数定义完成（避免缓冲区限制）")


✅ HTTP Stream 执行函数定义完成（避免缓冲区限制）


## 5. 查询所有 Room ID 和版本范围


In [4]:
# 查询所有 room-id 及版本范围
print("🔍 正在查询数据库...")
room_list = get_all_room_ids_with_versions()

if room_list:
    df = pd.DataFrame(room_list)
    df.index = df.index + 1
    
    print(f"\n✅ 找到 {len(room_list)} 个 Pad\n")
    display(df)
    
    # 统计信息
    total_revisions = df['revision_count'].sum()
    avg_revisions = df['revision_count'].mean()
    
    print(f"\n📊 统计信息:")
    print(f"   总 Pad 数: {len(room_list)}")
    print(f"   总版本数: {total_revisions}")
    print(f"   平均版本数: {avg_revisions:.2f}")
    print(f"   最多版本: {df['revision_count'].max()} (Pad: {df.loc[df['revision_count'].idxmax(), 'pad_id']})")
    print(f"   最少版本: {df['revision_count'].min()} (Pad: {df.loc[df['revision_count'].idxmin(), 'pad_id']})")
else:
    print("❌ 未找到任何 Pad 数据")
    df = pd.DataFrame()


🔍 正在查询数据库...

✅ 找到 13 个 Pad



,pad_id,min_revision,max_revision,revision_count
1,room-165,0,0,1
2,room-207,0,0,1
3,room-208,0,24,25
4,room-210,0,0,1
5,room-224,0,0,1
6,room-229,0,200,201
7,room-243,0,29,30
8,room-254,0,0,1
9,room-255,0,0,1
10,room-258,0,8,9



📊 统计信息:
   总 Pad 数: 13
   总版本数: 1244
   平均版本数: 95.69
   最多版本: 735 (Pad: room-262)
   最少版本: 1 (Pad: room-165)


## 6. 执行 Pad 内容重建

💡 **使用说明：** 修改下面代码中的参数，然后运行单元格

- `pad_id`: 从上面表格中选择一个 Pad ID
- `start_rev`: 起始版本号
- `end_rev`: 结束版本号


In [8]:
# ========================================
# 手动设置参数并执行重建
# ========================================

# 📝 在这里修改参数
pad_id = 'room-262'        # 修改为你要查询的 Pad ID
start_rev = 0              # 起始版本号
end_rev = 734              # 结束版本号
use_base64 = True          # 使用 Base64 编码模式（推荐用于大数据量和远程调用）
batch_size = 1            # 每次请求的版本数量（避免数据量过大导致解析失败）

# 执行重建
if not df.empty:
    print(f"\n{'='*80}")
    print(f"📝 Pad ID: {pad_id}")
    print(f"📊 版本范围: {start_rev} - {end_rev}")
    print(f"🔧 编码模式: {'Base64（推荐）' if use_base64 else '标准模式'}")
    print(f"📦 批次大小: 每次请求 {batch_size} 个版本")
    print(f"⏰ 开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}\n")
    
    # 计算需要的批次数
    total_versions = end_rev - start_rev + 1
    num_batches = (total_versions + batch_size - 1) // batch_size
    
    print(f"📊 总共需要 {num_batches} 个批次来处理 {total_versions} 个版本\n")
    
    # 初始化合并结果
    all_versions = []
    total_success = 0
    total_failed = 0
    
    # 分批执行重建
    for batch_idx in range(num_batches):
        batch_start = start_rev + batch_idx * batch_size
        batch_end = min(batch_start + batch_size - 1, end_rev)
        
        print(f"🔄 批次 {batch_idx + 1}/{num_batches}: 版本 {batch_start} - {batch_end}")
        
        # 执行当前批次的重建
        batch_result = execute_rebuild_script(pad_id, batch_start, batch_end, use_base64=use_base64)
        
        if batch_result.get('success'):
            batch_versions = batch_result.get('versions', [])
            all_versions.extend(batch_versions)
            
            batch_stats = batch_result.get('statistics', {})
            batch_success = batch_stats.get('success', 0)
            batch_failed = batch_stats.get('failed', 0)
            
            total_success += batch_success
            total_failed += batch_failed
            
            print(f"   ✅ 成功: {batch_success}, 失败: {batch_failed}")
        else:
            print(f"   ❌ 批次失败: {batch_result.get('error', 'Unknown error')}")
            total_failed += (batch_end - batch_start + 1)
        
        print()
    
    # 构建最终结果
    rebuild_result = {
        'success': True,
        'pad_id': pad_id,
        'head_revision': end_rev,
        'requested_range': {
            'start': start_rev,
            'end': end_rev
        },
        'statistics': {
            'total': total_versions,
            'success': total_success,
            'failed': total_failed
        },
        'versions': all_versions,
        'batch_info': {
            'batch_size': batch_size,
            'num_batches': num_batches
        }
    }
    
    print(f"{'='*80}")
    print(f"✅ 所有批次重建完成！")
    print(f"{'='*80}")
    print(f"   总版本数: {total_versions}")
    print(f"   成功: {total_success}")
    print(f"   失败: {total_failed}")
    print(f"   批次数: {num_batches}")
    print(f"\n⏰ 完成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\n💡 结果已保存到变量 rebuild_result，可以在下面的单元格中查看详细内容")
else:
    print("❌ 没有可用的 Pad 数据，请先执行上面的查询")


📝 Pad ID: room-262
📊 版本范围: 0 - 734
🔧 编码模式: Base64（推荐）
📦 批次大小: 每次请求 1 个版本
⏰ 开始时间: 2026-01-19 19:58:43

📊 总共需要 735 个批次来处理 735 个版本

🔄 批次 1/735: 版本 0 - 0
🌐 HTTP Stream 请求: http://localhost:9002/api/rebuild?padId=room-262&useBase64=true&startRev=0&endRev=0
📦 使用 Base64 编码模式
✅ HTTP 连接成功 (状态码: 200)
📊 处理范围: {'start': 0, 'end': 0}
🔓 解码 Base64 数据...
✅ Base64 解码完成
✅ 成功获取 1 个版本数据
   ✅ 成功: 0, 失败: 1

🔄 批次 2/735: 版本 1 - 1
🌐 HTTP Stream 请求: http://localhost:9002/api/rebuild?padId=room-262&useBase64=true&startRev=1&endRev=1
📦 使用 Base64 编码模式
✅ HTTP 连接成功 (状态码: 200)
📊 处理范围: {'start': 1, 'end': 1}
🔓 解码 Base64 数据...
✅ Base64 解码完成
✅ 成功获取 0 个版本数据
   ✅ 成功: 0, 失败: 0

🔄 批次 3/735: 版本 2 - 2
🌐 HTTP Stream 请求: http://localhost:9002/api/rebuild?padId=room-262&useBase64=true&startRev=2&endRev=2
📦 使用 Base64 编码模式
✅ HTTP 连接成功 (状态码: 200)
📊 处理范围: {'start': 2, 'end': 2}
🔓 解码 Base64 数据...
✅ Base64 解码完成
✅ 成功获取 0 个版本数据
   ✅ 成功: 0, 失败: 0

🔄 批次 4/735: 版本 3 - 3
🌐 HTTP Stream 请求: http://localhost:9002/api/rebuild?padId=room-262&us

In [ ]:
## 7. 版本详情查看器 - 查看对应版本的内容

In [11]:
rebuild_result

{'success': True,
 'pad_id': 'room-262',
 'head_revision': 349,
 'requested_range': {'start': 300, 'end': 349},
 'statistics': {'total': 50, 'success': 9, 'failed': 1},
 'versions': [{'revision': 300,
   'success': False,
   'error': 'pool.getAttrib is not a function'},
  {'revision': 301,
   'success': True,
   'pad_id': 'room-262',
   'author': 'a.nFhz3hmAId37Non8',
   'timestamp': 1766577546666,
   'formatted_timestamp': '2025-12-24 19:59:06.666',
   'text_length': 1043,
   'line_count': 25,
   'change_summary': '1041 -> 1044 chars',
   'content': 'Welcome to Etherpad!\n\nThis pad text is synchronized as you type, so that everyone viewing this page sees the same text. This allows you to collaborate seamlessly on documents!\n\nGet involved with Etherpad at https://etherpad.org\n\n杨沁：\n（1）我认为大模型可以针对问题给出更精确的回答，但是搜索引擎给出的范围比较广。\n（2）大模型会给出虚假的信息，会迎合用户的偏好，但是搜索引擎的内容就比较客观\n王紫娟：①我感觉可以向大模型说出自己的想法，大模型会根据用户想法提供反馈，但是搜索引擎只能根据部分关键词进行搜索；②但是向大模型提问一般问题还可以，提供学术方面的有时候会提供幻觉信息，这个目前比较让人苦恼；③大模型提供的内容更有条理性，搜索引

In [8]:
# 取出版本列表
versions = rebuild_result["versions"]

# 转 DataFrame
revision_content = pd.DataFrame([
    {"revision": v["revision"], "content": v["content"]}
    for v in versions
])

revision_content

KeyError: 'content'

## 8. 查看具体版本的详细内容

修改 `view_revision` 参数来查看不同版本


In [ ]:
# 📝 修改这里的版本号
view_revision = 5

# 查看版本详情
if 'rebuild_result' in globals() and rebuild_result and rebuild_result.get('success'):
    versions = rebuild_result.get('versions', [])
    version = next((v for v in versions if v.get('revision') == view_revision), None)
    
    if version and version.get('success'):
        print(f"\n{'='*80}")
        print(f"📝 版本 {view_revision} 详细信息")
        print(f"{'='*80}")
        print(f"Pad ID:        {version.get('pad_id', 'N/A')}")
        print(f"作者:          {version.get('author', 'N/A')}")
        print(f"时间戳:        {version.get('timestamp', 'N/A')}")
        print(f"格式化时间:    {version.get('formatted_timestamp', 'N/A')}")
        print(f"文本长度:      {version.get('text_length', 0)} 字符")
        print(f"行数:          {version.get('line_count', 0)}")
        print(f"变更摘要:      {version.get('change_summary', 'N/A')}")
        
        print(f"\n{'='*80}")
        print(f"📄 文本内容")
        print(f"{'='*80}")
        print(version.get('content', ''))
        
        print(f"\n{'='*80}")
        print(f"🔧 Changeset")
        print(f"{'='*80}")
        print(version.get('changeset', 'N/A'))
        
        print(f"\n{'='*80}")
        print(f"🎨 Attributes")
        print(f"{'='*80}")
        attribs = version.get('attribs', '')
        print(attribs if attribs else '(无属性)')
    else:
        print(f"❌ 未找到版本 {view_revision}")
else:
    print("⚠️ 请先执行重建")



📝 版本 5 详细信息
Pad ID:        room-229
作者:          a.ni6xvsCFoJs9Rr1v
时间戳:        1758377771664
格式化时间:    2025-09-20 22:16:11.664
文本长度:      37 字符
行数:          4
变更摘要:      30 -> 38 chars

📄 文本内容
Welcome to Etherpad!

周末去哪里玩
可以去迪士尼二元

🔧 Changeset
Z:u>8|3=t*0+8$可以去迪士尼二元

🎨 Attributes
|2+m*0|1+7*0+8|1+1


## 9. 读取 pad_version_changes 表数据

从数据库中读取指定 pad 的版本变化记录

In [ ]:
def get_pad_version_changes(room_id=None):
    """从数据库中读取指定 pad 的版本变化记录"""
    try:
        connection = pymysql.connect(**DB_CONFIG)
        cursor = connection.cursor(pymysql.cursors.DictCursor)
        
        if room_id:
            query = """SELECT * FROM pad_version_changes WHERE pad_id = %s ORDER BY seq_order ASC"""
            cursor.execute(query, [room_id])
        else:
            query = """SELECT * FROM pad_version_changes ORDER BY pad_id, seq_order ASC"""
            cursor.execute(query)
        
        results = cursor.fetchall()
        
        cursor.close()
        connection.close()
        
        return results
        
    except Exception as e:
        print(f"❌ 数据库查询失败: {e}")
        return []

print("✅ 数据库查询函数定义完成")

✅ 数据库查询函数定义完成


In [ ]:
# 读取数据
room_id = "room-229"
df_versions = get_pad_version_changes(room_id=room_id)
df_versions = pd.DataFrame(df_versions)
if df_versions is not None and len(df_versions) > 0:
    df = df_versions[[
    "pad_id",
    "seq_order",
    "behavior",
    "author",
    "content",
    "add_start_time",
    "add_end_time",
    "delete_start_time",
    "delete_end_time"
]]

df_versions.head(5)

,pad_id,seq_order,behavior,author,content,add_start_time,add_end_time,delete_start_time,delete_end_time
0,room-229,1,deleted,a.rVEwX679hNTTNivd,*,2025-09-26 23:05:32.012,2025-09-26 23:05:32.012,2025-10-21 22:47:16.498,2025-10-21 22:47:16.498
1,room-229,2,add,a.rVEwX679hNTTNivd,欢迎来到,2025-10-21 22:47:34.952,2025-10-21 22:47:34.952,None,None
2,room-229,3,add,a.ni6xvsCFoJs9Rr1v,Welcome to Etherpad!\n\n,2025-09-17 23:39:12.431,2025-09-17 23:39:12.431,None,None
3,room-229,4,deleted,a.ni6xvsCFoJs9Rr1v,"This pad text is synchronized as you type, so ...",2025-09-17 23:39:12.431,2025-09-17 23:39:12.431,2025-09-20 22:15:52.139,2025-09-20 22:15:52.139
4,room-229,5,add,a.ni6xvsCFoJs9Rr1v,周末去哪里玩\n可以去迪士尼,2025-09-20 22:15:58.322,2025-09-20 22:16:11.664,None,None
